# ECG Classification: MAE vs SAE vs PCA — PTB-XL Benchmark

**Mohammed Irshad Kunnam Puthoor** | MSc Applied Informatics | VMU 2026  
**Supervisor:** Prof. Ausra Saudargiene

Comparative study of Masked Autoencoder (MAE), Standard Autoencoder (SAE), and PCA feature extraction across three deployment tiers — Efficient Scan, Pretrained Fine-Tuned, and Full Analysis — on the full PTB-XL dataset (21,430 records, 5 classes).

In [ ]:
!pip install wfdb xgboost psutil scikit-posthocs -q

In [ ]:
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import wfdb
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import scikit_posthocs as sp

from scipy.signal import butter, filtfilt
from scipy.stats import kruskal
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, roc_curve, auc, classification_report
)
from tensorflow.keras import layers, Model, Input, optimizers, callbacks
from xgboost import XGBClassifier

np.random.seed(42)
tf.random.set_seed(42)

print(f'TensorFlow {tf.__version__} | GPU: {len(tf.config.list_physical_devices("GPU")) > 0}')

In [ ]:
DATA_DIR  = '/kaggle/input/datasets/khyeh0719/ptb-xl-dataset/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1'
CSV_PATH  = os.path.join(DATA_DIR, 'ptbxl_database.csv')
SCP_PATH  = os.path.join(DATA_DIR, 'scp_statements.csv')
CACHE_DIR = '/kaggle/working'

TOTAL_RECORDS_TO_LOAD = None  # None = full dataset
MAX_SAMPLES           = 1000
CHANNELS              = 12

LATENT_DIM = 128
PATCH_SIZE = 25
MASK_RATIO = 0.50
EPOCHS_AE  = 50
EPOCHS_DL  = 30
BATCH_SIZE = 64
TEST_SIZE  = 0.2

EMBED_DIM              = 64
NUM_HEADS              = 4
FF_DIM                 = 128
NUM_TRANSFORMER_BLOCKS = 2

SVM_C        = 1.0
XGB_N_EST    = 100
XGB_DEPTH    = 6
BILSTM_UNITS = 64
DROPOUT      = 0.3
RANDOM_STATE = 42

CV_FOLDS_ML   = 10
CV_REPEATS_ML = 5
CV_FOLDS_DL   = 5
CV_REPEATS_DL = 2

LABEL_MAP   = {'NORM': 0, 'MI': 1, 'STTC': 2, 'HYP': 3, 'CD': 4}
CLASS_NAMES = list(LABEL_MAP.keys())

def log(msg): print(f'[INFO] {msg}')

In [ ]:
CACHE_FILES = {k: os.path.join(CACHE_DIR, f'{k}.npy')
               for k in ['X_train', 'X_test', 'y_train', 'y_test']}

def bandpass_filter(data, lowcut=0.5, highcut=40.0, fs=100.0, order=3):
    nyq  = 0.5 * fs
    b, a = butter(order, [lowcut / nyq, highcut / nyq], btype='band')
    return filtfilt(b, a, data, axis=0)

def load_ptbxl():
    log('Loading PTB-XL metadata...')
    df     = pd.read_csv(CSV_PATH, index_col='ecg_id')
    agg_df = pd.read_csv(SCP_PATH, index_col=0)

    diag_col = next(c for c in ['diagnostic_class', 'diagnostic_superclass', 'diagnostic']
                    if c in agg_df.columns)
    agg_df = agg_df[agg_df[diag_col].notnull()]

    def get_label(y_dic):
        if isinstance(y_dic, str): y_dic = eval(y_dic)
        labels = list(set(agg_df.loc[k][diag_col] for k in y_dic if k in agg_df.index))
        return labels[0] if labels else None

    df['label']    = df.scp_codes.apply(get_label)
    df             = df.dropna(subset=['label'])
    df             = df[df['label'].isin(LABEL_MAP)]
    df['label_id'] = df['label'].map(LABEL_MAP)

    n_total = len(df)
    n_load  = min(TOTAL_RECORDS_TO_LOAD, n_total) if TOTAL_RECORDS_TO_LOAD else n_total
    df      = df.sample(n=n_load, random_state=RANDOM_STATE)
    log(f'Total valid records: {n_total} | Loading: {n_load}')

    X, y, count = np.zeros((n_load, MAX_SAMPLES, CHANNELS), dtype=np.float32), np.zeros(n_load, dtype=np.int32), 0
    for i, (_, row) in enumerate(df.iterrows()):
        if i % 2000 == 0: log(f'  {i}/{n_load}')
        try:
            sig, _ = wfdb.rdsamp(os.path.join(DATA_DIR, row['filename_lr']))
            if sig.shape[1] >= 12 and len(sig) >= MAX_SAMPLES:
                X[count] = bandpass_filter(sig[:MAX_SAMPLES]).astype(np.float32)
                y[count] = row['label_id']
                count   += 1
        except Exception:
            continue

    log(f'Loaded {count} records')
    return X[:count], y[:count]

if all(os.path.exists(v) for v in CACHE_FILES.values()):
    log('Cache found — loading arrays...')
    X_train, X_test = np.load(CACHE_FILES['X_train']), np.load(CACHE_FILES['X_test'])
    y_train, y_test = np.load(CACHE_FILES['y_train']), np.load(CACHE_FILES['y_test'])
else:
    X, y = load_ptbxl()
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
    )
    scaler  = StandardScaler()
    X_train = scaler.fit_transform(X_train.reshape(-1, CHANNELS)).reshape(X_train.shape).astype(np.float32)
    X_test  = scaler.transform(X_test.reshape(-1, CHANNELS)).reshape(X_test.shape).astype(np.float32)
    for k, arr in zip(CACHE_FILES, [X_train, X_test, y_train, y_test]):
        np.save(CACHE_FILES[k], arr)
    log('Cached to disk.')

log(f'Train: {X_train.shape} | Test: {X_test.shape}')
for u, c in zip(*np.unique(y_train, return_counts=True)):
    log(f'  {CLASS_NAMES[u]}: {c} ({c/len(y_train):.1%})')

CLASS_WEIGHT_ARR = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
CLASS_WEIGHT     = dict(enumerate(CLASS_WEIGHT_ARR))
X_all = np.concatenate([X_train, X_test])
y_all = np.concatenate([y_train, y_test])

In [ ]:
def apply_mae_mask(X, seed=RANDOM_STATE):
    """Fixed-seed masking — same pattern every call (used for validation and feature extraction)."""
    X_masked    = X.copy()
    num_patches = X.shape[1] // PATCH_SIZE
    num_mask    = int(num_patches * MASK_RATIO)
    rng         = np.random.RandomState(seed)
    for i in range(len(X)):
        for idx in rng.choice(num_patches, num_mask, replace=False):
            X_masked[i, idx * PATCH_SIZE:(idx + 1) * PATCH_SIZE] = 0.0
    return X_masked

def apply_mae_mask_dynamic(X):
    """Random masking — new mask per call (used during MAE training for each epoch)."""
    X_masked    = X.copy()
    num_patches = X.shape[1] // PATCH_SIZE
    num_mask    = int(num_patches * MASK_RATIO)
    for i in range(len(X)):
        for idx in np.random.choice(num_patches, num_mask, replace=False):
            X_masked[i, idx * PATCH_SIZE:(idx + 1) * PATCH_SIZE] = 0.0
    return X_masked

X_train_masked = apply_mae_mask(X_train)
log(f'MAE masking: {int(MASK_RATIO*100)}% of {X_train.shape[1]//PATCH_SIZE} patches zeroed per record')

In [ ]:
class _PosEmbed(layers.Layer):
    """Learnable positional embedding for patch token sequences."""
    def __init__(self, num_patches, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.num_patches = num_patches
        self.embed_dim   = embed_dim
        self.emb         = layers.Embedding(num_patches, embed_dim)

    def call(self, x):
        return x + self.emb(tf.range(self.num_patches))

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'num_patches': self.num_patches, 'embed_dim': self.embed_dim})
        return cfg


def build_autoencoder(input_shape=(MAX_SAMPLES, CHANNELS)):
    num_patches = input_shape[0] // PATCH_SIZE

    enc_in = Input(shape=input_shape)
    x = layers.Reshape((num_patches, PATCH_SIZE * CHANNELS))(enc_in)
    x = layers.Dense(EMBED_DIM)(x)
    x = _PosEmbed(num_patches, EMBED_DIM)(x)
    for _ in range(NUM_TRANSFORMER_BLOCKS):
        attn = layers.MultiHeadAttention(num_heads=NUM_HEADS, key_dim=EMBED_DIM // NUM_HEADS, dropout=0.1)(x, x)
        x    = layers.LayerNormalization(epsilon=1e-6)(x + attn)
        ff   = layers.Dense(FF_DIM, activation='gelu')(x)
        ff   = layers.Dense(EMBED_DIM)(ff)
        x    = layers.LayerNormalization(epsilon=1e-6)(x + ff)
    x      = layers.GlobalAveragePooling1D()(x)
    latent = layers.Dense(LATENT_DIM, activation='relu', name='latent_features')(x)
    encoder = Model(enc_in, latent, name='Encoder')

    dec_in = Input(shape=(LATENT_DIM,))
    x      = layers.Dense(num_patches * EMBED_DIM, activation='gelu')(dec_in)
    x      = layers.Reshape((num_patches, EMBED_DIM))(x)
    x      = layers.Dense(PATCH_SIZE * CHANNELS, activation='linear')(x)
    out    = layers.Reshape(input_shape)(x)
    decoder = Model(dec_in, out, name='Decoder')

    ae = Model(enc_in, decoder(encoder(enc_in)), name='Autoencoder')
    ae.compile(optimizer=optimizers.Adam(0.001), loss='mse')
    return ae, encoder


def build_finetuned_classifier(encoder_path, num_classes=5, lr=1e-4):
    encoder = tf.keras.models.load_model(encoder_path, custom_objects={'_PosEmbed': _PosEmbed})
    for layer in encoder.layers:
        layer.trainable = True
    # Explicit names prevent layer name collision after tf.keras.backend.clear_session()
    x   = layers.Dense(64, activation='relu', name='clf_dense')(encoder.output)
    x   = layers.Dropout(DROPOUT, name='clf_dropout')(x)
    out = layers.Dense(num_classes, activation='softmax', name='clf_output')(x)
    m   = Model(encoder.input, out)
    m.compile(optimizer=optimizers.Adam(lr), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m


def build_bilstm(num_classes=5):
    inp = Input(shape=(MAX_SAMPLES, CHANNELS))
    x   = layers.Bidirectional(layers.LSTM(BILSTM_UNITS, return_sequences=True))(inp)
    x   = layers.GlobalAveragePooling1D()(x)
    x   = layers.Dropout(DROPOUT)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)
    m   = Model(inp, out)
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m


def build_cnn(num_classes=5):
    inp = Input(shape=(MAX_SAMPLES, CHANNELS))
    x   = layers.Conv1D(32,  7, activation='relu', padding='same')(inp)
    x   = layers.MaxPooling1D(2)(x)
    x   = layers.Conv1D(64,  5, activation='relu', padding='same')(x)
    x   = layers.MaxPooling1D(2)(x)
    x   = layers.Conv1D(128, 3, activation='relu', padding='same')(x)
    x   = layers.GlobalAveragePooling1D()(x)
    x   = layers.Dropout(DROPOUT)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)
    m   = Model(inp, out)
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

In [ ]:
es_ae = callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)

log('Training SAE...')
sae, encoder_sae = build_autoencoder()
history_sae = sae.fit(X_train, X_train, epochs=EPOCHS_AE, batch_size=BATCH_SIZE,
                      validation_split=0.1, callbacks=[es_ae], verbose=1)

log('Training MAE with dynamic re-masking...')
mae, encoder_mae = build_autoencoder()

n_val        = int(len(X_train) * 0.1)
X_tr_ae      = X_train[:-n_val]
X_val_ae     = X_train[-n_val:]
X_val_masked = apply_mae_mask(X_val_ae)

best_val, patience_counter   = np.inf, 0
mae_train_hist, mae_val_hist = [], []

for epoch in range(EPOCHS_AE):
    h = mae.fit(apply_mae_mask_dynamic(X_tr_ae), X_tr_ae,
                epochs=1, batch_size=BATCH_SIZE,
                validation_data=(X_val_masked, X_val_ae), verbose=1)
    tl, vl = h.history['loss'][0], h.history['val_loss'][0]
    mae_train_hist.append(tl)
    mae_val_hist.append(vl)
    if vl < best_val:
        best_val, patience_counter = vl, 0
        mae.save_weights('mae_best.weights.h5')
    else:
        patience_counter += 1
    if patience_counter >= 5:
        log(f'Early stopping at epoch {epoch + 1}')
        break

mae.load_weights('mae_best.weights.h5')

class _Hist:
    def __init__(self, loss, val_loss):
        self.history = {'loss': loss, 'val_loss': val_loss}

history_mae = _Hist(mae_train_hist, mae_val_hist)

encoder_sae.save('encoder_sae.keras')
encoder_mae.save('encoder_mae.keras')
log('Encoders saved.')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for ax, hist, title in [(ax1, history_sae, 'SAE'), (ax2, history_mae, 'MAE')]:
    ax.plot(hist.history['loss'],     label='Train', color='steelblue')
    ax.plot(hist.history['val_loss'], label='Validation', color='crimson')
    ax.set_title(f'{title} Training Curves', fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Autoencoder Convergence — SAE vs MAE', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
log('Extracting 128-dim latent features...')

X_train_sae = encoder_sae.predict(X_train, verbose=0)
X_test_sae  = encoder_sae.predict(X_test,  verbose=0)
X_train_mae = encoder_mae.predict(X_train, verbose=0)
X_test_mae  = encoder_mae.predict(X_test,  verbose=0)

pca         = PCA(n_components=LATENT_DIM)
X_train_pca = pca.fit_transform(X_train.reshape(len(X_train), -1))
X_test_pca  = pca.transform(X_test.reshape(len(X_test), -1))

X_all_pca = np.concatenate([X_train_pca, X_test_pca])
X_all_sae = np.concatenate([X_train_sae, X_test_sae])
X_all_mae = np.concatenate([X_train_mae, X_test_mae])

np.save('X_test_pca.npy', X_test_pca)
np.save('X_test_sae.npy', X_test_sae)
np.save('X_test_mae.npy', X_test_mae)

log(f'Feature shape: {X_all_sae.shape} (identical across PCA / SAE / MAE)')

In [ ]:
sns.set_theme(style='whitegrid')

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
ax1.plot(X_train[0, :, 0], color='steelblue', linewidth=0.8)
ax1.set_title('Original ECG Signal (Lead I)', fontweight='bold')
ax1.set_ylabel('Amplitude')
ax2.plot(X_train_masked[0, :, 0], color='crimson', linewidth=0.8)
ax2.set_title(f'Masked Input — {int(MASK_RATIO*100)}% of patches zeroed during pretraining', fontweight='bold')
ax2.set_ylabel('Amplitude')
ax2.set_xlabel('Timestep (100 Hz)')
plt.suptitle('MAE Self-Supervised Pretraining Strategy', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs_mae_masking.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
reconstructed = mae.predict(X_train_masked[0:1], verbose=0)[0]

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
axes[0].plot(X_train[0, :, 0],        color='steelblue',   linewidth=0.8)
axes[0].set_title('Original Signal', fontweight='bold')
axes[1].plot(X_train_masked[0, :, 0], color='crimson',     linewidth=0.8)
axes[1].set_title(f'Masked Input ({int(MASK_RATIO*100)}% hidden)', fontweight='bold')
axes[2].plot(reconstructed[:, 0],     color='forestgreen', linewidth=0.8)
axes[2].set_title('MAE Reconstruction', fontweight='bold')
for ax in axes: ax.set_ylabel('Amplitude')
axes[2].set_xlabel('Timestep')
plt.suptitle('MAE Reconstruction Quality — Pretraining Verification', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs_reconstruction.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
log(f'ML CV: {CV_FOLDS_ML}-fold × {CV_REPEATS_ML} repeats ({CV_FOLDS_ML * CV_REPEATS_ML} measurements per model)')

ml_cv_results  = {}
ml_lat_results = {}
rskf = RepeatedStratifiedKFold(n_splits=CV_FOLDS_ML, n_repeats=CV_REPEATS_ML, random_state=RANDOM_STATE)

for name, X_feat, clf_type in [
    ('PCA + SVM',     X_all_pca, 'svm'),
    ('SAE + SVM',     X_all_sae, 'svm'),
    ('MAE + SVM',     X_all_mae, 'svm'),
    ('MAE + XGBoost', X_all_mae, 'xgb'),
]:
    accs, lats = [], []
    for fold_i, (tr, te) in enumerate(rskf.split(X_feat, y_all)):
        if clf_type == 'svm':
            clf = CalibratedClassifierCV(LinearSVC(C=SVM_C, max_iter=3000, class_weight='balanced'))
            clf.fit(X_feat[tr], y_all[tr])
        else:
            sw  = np.array([CLASS_WEIGHT_ARR[c] for c in y_all[tr]])
            clf = XGBClassifier(n_estimators=XGB_N_EST, max_depth=XGB_DEPTH,
                                eval_metric='mlogloss', n_jobs=-1, random_state=RANDOM_STATE)
            clf.fit(X_feat[tr], y_all[tr], sample_weight=sw)
        t0 = time.time()
        preds = clf.predict(X_feat[te])
        lats.append((time.time() - t0) * 1000)
        accs.append(accuracy_score(y_all[te], preds))
        if (fold_i + 1) % 10 == 0:
            log(f'  {name} fold {fold_i+1:02d}/{CV_FOLDS_ML*CV_REPEATS_ML} | mean: {np.mean(accs):.4f}')
    ml_cv_results[name]  = accs
    ml_lat_results[name] = lats
    log(f'{name}: {np.mean(accs):.4f} ± {np.std(accs):.4f}')

log('ML CV complete')

In [ ]:
log(f'DL CV: {CV_FOLDS_DL}-fold × {CV_REPEATS_DL} repeats ({CV_FOLDS_DL * CV_REPEATS_DL} measurements per model)')

dl_cv_results  = {}
dl_lat_results = {}
rskf_dl = RepeatedStratifiedKFold(n_splits=CV_FOLDS_DL, n_repeats=CV_REPEATS_DL, random_state=RANDOM_STATE)
es_dl   = callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True, verbose=0)

for model_name, enc_path in [('SAE Fine-tuned', 'encoder_sae.keras'), ('MAE Fine-tuned', 'encoder_mae.keras')]:
    accs, lats = [], []
    for fold_i, (tr, te) in enumerate(rskf_dl.split(X_all, y_all)):
        tf.keras.backend.clear_session()
        cw    = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_all[tr]), y=y_all[tr])))
        model = build_finetuned_classifier(enc_path)
        model.fit(X_all[tr], y_all[tr], epochs=EPOCHS_DL, batch_size=BATCH_SIZE,
                  validation_split=0.1, callbacks=[es_dl], class_weight=cw, verbose=0)
        t0 = time.time()
        probs = model.predict(X_all[te], verbose=0)
        lats.append((time.time() - t0) * 1000)
        accs.append(accuracy_score(y_all[te], np.argmax(probs, axis=1)))
        log(f'  {model_name} fold {fold_i+1:02d}/{CV_FOLDS_DL*CV_REPEATS_DL} — acc: {accs[-1]:.4f}')
    dl_cv_results[model_name]  = accs
    dl_lat_results[model_name] = lats
    log(f'{model_name}: {np.mean(accs):.4f} ± {np.std(accs):.4f}')

for model_name, build_fn in [('Raw + BiLSTM', build_bilstm), ('Raw + 1D-CNN', build_cnn)]:
    accs, lats = [], []
    for fold_i, (tr, te) in enumerate(rskf_dl.split(X_all, y_all)):
        tf.keras.backend.clear_session()
        cw    = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_all[tr]), y=y_all[tr])))
        model = build_fn()
        model.fit(X_all[tr], y_all[tr], epochs=EPOCHS_DL, batch_size=BATCH_SIZE,
                  validation_split=0.1, callbacks=[es_dl], class_weight=cw, verbose=0)
        t0 = time.time()
        probs = model.predict(X_all[te], verbose=0)
        lats.append((time.time() - t0) * 1000)
        accs.append(accuracy_score(y_all[te], np.argmax(probs, axis=1)))
        log(f'  {model_name} fold {fold_i+1:02d}/{CV_FOLDS_DL*CV_REPEATS_DL} — acc: {accs[-1]:.4f}')
    dl_cv_results[model_name]  = accs
    dl_lat_results[model_name] = lats
    log(f'{model_name}: {np.mean(accs):.4f} ± {np.std(accs):.4f}')

log('DL CV complete')

In [ ]:
all_cv_results  = {**ml_cv_results,  **dl_cv_results}
all_lat_results = {**ml_lat_results, **dl_lat_results}

model_names = [
    'PCA + SVM', 'SAE + SVM', 'MAE + SVM', 'MAE + XGBoost',
    'SAE Fine-tuned', 'MAE Fine-tuned', 'Raw + BiLSTM', 'Raw + 1D-CNN',
]
MODE_MAP = {
    'PCA + SVM': 'Efficient Scan', 'SAE + SVM': 'Efficient Scan',
    'MAE + SVM': 'Efficient Scan', 'MAE + XGBoost': 'Efficient Scan',
    'SAE Fine-tuned': 'Pretrained FT', 'MAE Fine-tuned': 'Pretrained FT',
    'Raw + BiLSTM': 'Full Analysis',  'Raw + 1D-CNN': 'Full Analysis',
}

summary_rows = []
print(f'\n{"Model":<22} {"Mode":<16} {"N":>4} {"Mean":>8} {"Std":>6} {"95% CI":>22} {"Latency":>10}')
print('─' * 98)
for name in model_names:
    accs, lats = all_cv_results[name], all_lat_results[name]
    n, mean, std = len(accs), np.mean(accs), np.std(accs)
    ci_lo, ci_hi = mean - 1.96 * std / np.sqrt(n), mean + 1.96 * std / np.sqrt(n)
    print(f'{name:<22} {MODE_MAP[name]:<16} {n:>4} {mean:>8.4f} {std:>6.4f} [{ci_lo:.4f}, {ci_hi:.4f}] {np.mean(lats):>8.1f}ms')
    summary_rows.append({'Model': name, 'Mode': MODE_MAP[name], 'N_folds': n,
                         'Mean_Accuracy': round(mean, 4), 'Std': round(std, 4),
                         'CI_95_low': round(ci_lo, 4), 'CI_95_high': round(ci_hi, 4),
                         'Mean_Latency_ms': round(np.mean(lats), 1)})

pd.DataFrame(summary_rows).to_csv('cv_results_summary.csv', index=False)
log('Saved: cv_results_summary.csv')

In [ ]:
log('Running Kruskal-Wallis test...')

score_lists = [all_cv_results[n] for n in model_names]
kw_stat, kw_p = kruskal(*score_lists)

print(f'\nKruskal-Wallis H={kw_stat:.4f}, p={kw_p:.6f}')
print('Significant (p < 0.05)' if kw_p < 0.05 else 'Not significant')
print('Note: ML=50, DL=10 measurements — unequal N acknowledged as limitation.')

if kw_p < 0.05:
    all_scores, all_groups = [], []
    for name in model_names:
        all_scores.extend(all_cv_results[name])
        all_groups.extend([name] * len(all_cv_results[name]))
    ph_df = pd.DataFrame({'score': all_scores, 'group': all_groups})
    dunn  = sp.posthoc_dunn(ph_df, val_col='score', group_col='group',
                             p_adjust='bonferroni')
    print('\nDunn post-hoc p-values (Bonferroni corrected):')
    print(dunn.round(4).to_string())
    dunn.to_csv('dunn_posthoc.csv')

    plt.figure(figsize=(10, 8))
    mask = np.eye(len(dunn), dtype=bool)
    sns.heatmap(dunn, annot=True, fmt='.3f', cmap='RdYlGn_r',
                vmin=0, vmax=0.1, mask=mask, linewidths=0.5,
                annot_kws={'size': 9})
    plt.title('Dunn Post-hoc p-values (Bonferroni corrected)\n'
              'Red < 0.05 = significant pairwise difference | ML=50, DL=10 measurements',
              fontweight='bold', fontsize=11)
    plt.tight_layout()
    plt.savefig('outputs_posthoc.png', dpi=150, bbox_inches='tight')
    plt.show()
    log('Saved: outputs_posthoc.png | dunn_posthoc.csv')

In [ ]:
import pickle

log('Training final models on full train/test split...')

final_preds   = {}
final_proba   = {}
final_latency = {}

SAVE_NAMES = {
    'PCA + SVM':      'model_pca_svm.pkl',
    'SAE + SVM':      'model_sae_svm.pkl',
    'MAE + SVM':      'model_mae_svm.pkl',
    'MAE + XGBoost':  'model_mae_xgboost.pkl',
    'SAE Fine-tuned': 'model_sae_finetuned.keras',
    'MAE Fine-tuned': 'model_mae_finetuned.keras',
    'Raw + BiLSTM':   'model_bilstm.keras',
    'Raw + 1D-CNN':   'model_cnn.keras',
}

for name, x_tr, x_te, clf_type in [
    ('PCA + SVM',     X_train_pca, X_test_pca, 'svm'),
    ('SAE + SVM',     X_train_sae, X_test_sae, 'svm'),
    ('MAE + SVM',     X_train_mae, X_test_mae, 'svm'),
    ('MAE + XGBoost', X_train_mae, X_test_mae, 'xgb'),
]:
    if clf_type == 'svm':
        clf = CalibratedClassifierCV(LinearSVC(C=SVM_C, max_iter=3000, class_weight='balanced'))
    else:
        clf = XGBClassifier(
            n_estimators=XGB_N_EST, max_depth=XGB_DEPTH,
            eval_metric='mlogloss', n_jobs=-1, random_state=RANDOM_STATE
        )
    if clf_type == 'xgb':
        sw = np.array([CLASS_WEIGHT_ARR[c] for c in y_train])
        clf.fit(x_tr, y_train, sample_weight=sw)
    else:
        clf.fit(x_tr, y_train)
    t0 = time.time()
    preds = clf.predict(x_te)
    final_latency[name] = (time.time() - t0) * 1000
    final_preds[name]   = preds
    final_proba[name]   = clf.predict_proba(x_te)
    pickle.dump(clf, open(SAVE_NAMES[name], 'wb'))
    log(f'  {name}: acc={accuracy_score(y_test, preds):.4f} | {final_latency[name]:.1f}ms')

es_final = callbacks.EarlyStopping(monitor='val_loss', patience=5,
                                   restore_best_weights=True)
cnn_final_model = None

for model_name, enc_path in [
    ('SAE Fine-tuned', 'encoder_sae.keras'),
    ('MAE Fine-tuned', 'encoder_mae.keras'),
]:
    tf.keras.backend.clear_session()
    model = build_finetuned_classifier(enc_path)
    model.fit(X_train, y_train,
              epochs=EPOCHS_DL, batch_size=BATCH_SIZE,
              validation_split=0.1, callbacks=[es_final],
              class_weight=CLASS_WEIGHT, verbose=1)
    t0    = time.time()
    probs = model.predict(X_test, verbose=0)
    final_latency[model_name] = (time.time() - t0) * 1000
    final_preds[model_name]   = np.argmax(probs, axis=1)
    final_proba[model_name]   = probs
    model.save(SAVE_NAMES[model_name])
    log(f'  {model_name}: acc={accuracy_score(y_test, final_preds[model_name]):.4f}')

for model_name, build_fn in [('Raw + BiLSTM', build_bilstm), ('Raw + 1D-CNN', build_cnn)]:
    tf.keras.backend.clear_session()
    model = build_fn()
    model.fit(X_train, y_train,
              epochs=EPOCHS_DL, batch_size=BATCH_SIZE,
              validation_split=0.1, callbacks=[es_final],
              class_weight=CLASS_WEIGHT, verbose=1)
    t0    = time.time()
    probs = model.predict(X_test, verbose=0)
    final_latency[model_name] = (time.time() - t0) * 1000
    final_preds[model_name]   = np.argmax(probs, axis=1)
    final_proba[model_name]   = probs
    if model_name == 'Raw + 1D-CNN':
        cnn_final_model = model
    model.save(SAVE_NAMES[model_name])
    log(f'  {model_name}: acc={accuracy_score(y_test, final_preds[model_name]):.4f}')

log('All final models trained and saved.')

In [ ]:
print()
print('=' * 115)
print(f'{"Model":<22} {"Mode":<16} {"CV Mean±Std":>14} {"Test Acc":>10}'
      f' {"Precision":>10} {"Recall":>8} {"Macro F1":>10} {"Latency":>12}')
print('=' * 115)

for name in model_names:
    cv_m  = np.mean(all_cv_results[name])
    cv_s  = np.std(all_cv_results[name])
    mode  = MODE_MAP[name]
    preds = final_preds[name]
    t_acc = accuracy_score(y_test, preds)
    prec  = precision_score(y_test, preds, average='macro', zero_division=0)
    rec   = recall_score(y_test, preds,    average='macro', zero_division=0)
    f1    = f1_score(y_test, preds,        average='macro', zero_division=0)
    lat   = final_latency[name]
    print(f'{name:<22} {mode:<16} {cv_m:.3f}±{cv_s:.3f}   {t_acc:>9.3f}'
          f'  {prec:>9.3f}  {rec:>7.3f}  {f1:>9.3f}  {lat:>10.1f}ms')

print('=' * 115)
best = max(model_names, key=lambda n: accuracy_score(y_test, final_preds[n]))
log(f'Best model: {best} — {accuracy_score(y_test, final_preds[best]):.4f}')

In [ ]:
import matplotlib.patches as mpatches

means  = [np.mean(all_cv_results[n]) for n in model_names]
stds   = [np.std(all_cv_results[n])  for n in model_names]

colors = []
for n in model_names:
    if n in ml_cv_results:
        colors.append('#e07b39')
    elif 'Fine-tuned' in n:
        colors.append('#2d9d8f')
    else:
        colors.append('#2d6a9f')

plt.figure(figsize=(16, 6))
bars = plt.bar(model_names, means, yerr=stds, capsize=6,
               color=colors, edgecolor='white', error_kw={'linewidth': 2})
for bar, m, s in zip(bars, means, stds):
    plt.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + s + 0.012,
             f'{m:.1%}\n±{s:.3f}', ha='center',
             fontweight='bold', fontsize=8)

p1 = mpatches.Patch(color='#e07b39', label='Efficient Scan — fixed 128-dim features')
p2 = mpatches.Patch(color='#2d9d8f', label='Pretrained FT — encoder fine-tuned end-to-end')
p3 = mpatches.Patch(color='#2d6a9f', label='Full Analysis — raw ECG, no pretraining')
plt.legend(handles=[p1, p2, p3], loc='upper left')
plt.title(
    'ECG Classification Accuracy — Cross-Validated (Mean ± Std)\n'
    f'ML: {CV_FOLDS_ML}-fold × {CV_REPEATS_ML} repeats | '
    f'DL: {CV_FOLDS_DL}-fold × {CV_REPEATS_DL} repeats | Full PTB-XL',
    fontsize=12, fontweight='bold'
)
plt.ylabel('Accuracy')
plt.ylim(0, 1.15)
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.savefig('outputs_accuracy_cv.png', dpi=150, bbox_inches='tight')
plt.show()
log('Saved: outputs_accuracy_cv.png')

In [ ]:
f1_matrix = [
    f1_score(y_test, final_preds[n], average=None,
             labels=list(range(5)), zero_division=0)
    for n in model_names
]
plt.figure(figsize=(11, 7))
sns.heatmap(np.array(f1_matrix), annot=True, fmt='.2f', cmap='YlOrRd',
            xticklabels=CLASS_NAMES, yticklabels=model_names,
            vmin=0, vmax=1, linewidths=0.5, annot_kws={'size': 11, 'weight': 'bold'})
plt.title('Per-Class F1 Score — Final Test Set', fontsize=13, fontweight='bold')
plt.xlabel('Cardiac Condition')
plt.ylabel('Model')
plt.tight_layout()
plt.savefig('outputs_f1_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
log('Saved: outputs_f1_heatmap.png')

In [ ]:
lat_vals = [final_latency[n] for n in model_names]

colors2 = []
for n in model_names:
    if n in ml_cv_results:
        colors2.append('#e07b39')
    elif 'Fine-tuned' in n:
        colors2.append('#2d9d8f')
    else:
        colors2.append('#2d6a9f')

_, ax = plt.subplots(figsize=(16, 5))
bars2 = ax.bar(model_names, lat_vals, color=colors2, edgecolor='white')
for bar, t in zip(bars2, lat_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f'{t:.1f}ms', ha='center', fontweight='bold', fontsize=9)
ax.set_title('Inference Latency per Model', fontsize=13, fontweight='bold')
ax.set_ylabel('Time (ms)')
ax.set_xticks(range(len(model_names)))
ax.set_xticklabels(model_names, rotation=25, ha='right')
plt.tight_layout()
plt.savefig('outputs_latency.png', dpi=150, bbox_inches='tight')
plt.show()
log('Saved: outputs_latency.png')

In [ ]:
colors3 = ['#e63946', '#457b9d', '#2a9d8f', '#e9c46a', '#f4a261']

_, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
for features, ax, title in [
    (X_test_sae, ax1, 'SAE Latent Space'),
    (X_test_mae, ax2, 'MAE Latent Space (Pre-trained)'),
]:
    reduced = TSNE(n_components=2, random_state=RANDOM_STATE,
                   perplexity=30, max_iter=500).fit_transform(features)
    for i, cls in enumerate(CLASS_NAMES):
        mask = y_test == i
        ax.scatter(reduced[mask, 0], reduced[mask, 1],
                   c=colors3[i], label=cls, alpha=0.6, s=20)
    ax.set_title(title, fontweight='bold')
    ax.legend(markerscale=2)

plt.suptitle('Latent Space t-SNE — SAE vs MAE Pre-trained Features',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs_tsne.png', dpi=150, bbox_inches='tight')
plt.show()
log('Saved: outputs_tsne.png')

In [ ]:
correct_mask = (final_preds['Raw + 1D-CNN'] == y_test)
norm_correct  = np.where((y_test == 0) & correct_mask)[0]
mi_correct    = np.where((y_test == 1) & correct_mask)[0]

sample_idx = int(mi_correct[0]) if len(mi_correct) > 0 else int(norm_correct[0])

x_tensor = tf.convert_to_tensor(X_test[sample_idx:sample_idx+1], dtype=tf.float32)
with tf.GradientTape() as tape:
    tape.watch(x_tensor)
    pred            = cnn_final_model(x_tensor, training=False)
    predicted_class = int(tf.argmax(pred[0]))
    score           = pred[0, predicted_class]

grads    = tape.gradient(score, x_tensor)[0].numpy()
saliency = np.mean(np.abs(grads), axis=1)
saliency = (saliency - saliency.min()) / (saliency.max() - saliency.min() + 1e-8)

true_cls = CLASS_NAMES[y_test[sample_idx]]
pred_cls = CLASS_NAMES[predicted_class]
conf     = float(tf.reduce_max(pred[0]).numpy())

_, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True,
                              gridspec_kw={'height_ratios': [3, 1]})
ax1.plot(X_test[sample_idx, :, 0], color='steelblue', linewidth=0.8)
ax1.set_title(
    f'XAI Gradient Saliency — 1D-CNN\n'
    f'True: {true_cls} | Predicted: {pred_cls} ✓ | Confidence: {conf:.1%}',
    fontweight='bold'
)
ax1.set_ylabel('ECG Amplitude (Lead I)')
ax2.imshow(saliency[np.newaxis, :], aspect='auto', cmap='hot',
           extent=[0, len(saliency), 0, 1])
ax2.set_title('Gradient Saliency — red = highest model attention', fontsize=9)
ax2.set_xlabel('Timestep (samples @ 100 Hz)')
plt.suptitle('Explainability (XAI) — Correctly Classified Sample',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs_saliency.png', dpi=150, bbox_inches='tight')
plt.show()
log(f'Saved: outputs_saliency.png | Sample: {true_cls} → {pred_cls}')

In [ ]:
p_matrix = [precision_score(y_test, final_preds[n], average=None,
            labels=list(range(5)), zero_division=0) for n in model_names]
r_matrix = [recall_score(y_test, final_preds[n], average=None,
            labels=list(range(5)), zero_division=0) for n in model_names]

_, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))
sns.heatmap(np.array(p_matrix), annot=True, fmt='.2f', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=model_names,
            vmin=0, vmax=1, ax=ax1, annot_kws={'size': 10, 'weight': 'bold'})
ax1.set_title('Per-Class Precision', fontweight='bold')

sns.heatmap(np.array(r_matrix), annot=True, fmt='.2f', cmap='Greens',
            xticklabels=CLASS_NAMES, yticklabels=model_names,
            vmin=0, vmax=1, ax=ax2, annot_kws={'size': 10, 'weight': 'bold'})
ax2.set_title('Per-Class Recall', fontweight='bold')

plt.suptitle('Precision & Recall — Final Test Set (with class weights)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs_precision_recall.png', dpi=150, bbox_inches='tight')
plt.show()
log('Saved: outputs_precision_recall.png')

In [ ]:
y_bin = label_binarize(y_test, classes=list(range(5)))

_, axes = plt.subplots(3, 3, figsize=(18, 14))
axes    = axes.flatten()

for i, name in enumerate(model_names):
    ax    = axes[i]
    proba = final_proba[name]
    for j, cls in enumerate(CLASS_NAMES):
        fpr, tpr, _ = roc_curve(y_bin[:, j], proba[:, j])
        roc_auc     = auc(fpr, tpr)
        ax.plot(fpr, tpr, linewidth=1.5, label=f'{cls} (AUC={roc_auc:.2f})')
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.4)
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('FPR')
    ax.set_ylabel('TPR')
    ax.legend(fontsize=8)

for j in range(len(model_names), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('AUC-ROC Curves — All Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs_roc.png', dpi=150, bbox_inches='tight')
plt.show()
log('Saved: outputs_roc.png')

In [ ]:
_, axes = plt.subplots(3, 3, figsize=(18, 14))
axes    = axes.flatten()

for i, name in enumerate(model_names):
    cm = confusion_matrix(y_test, final_preds[name])
    sns.heatmap(cm, annot=True, fmt='d', ax=axes[i], cmap='Blues', cbar=False,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                annot_kws={'size': 10})
    acc = accuracy_score(y_test, final_preds[name])
    axes[i].set_title(f'{name}  |  Acc: {acc:.1%}', fontweight='bold', fontsize=10)
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

for j in range(len(model_names), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Confusion Matrices — All Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs_confusion.png', dpi=150, bbox_inches='tight')
plt.show()
log('Saved: outputs_confusion.png')

In [ ]:
print('\n' + '=' * 70)
print('  PER-CLASS REPORTS (Final Test Set)')
print('=' * 70)
for name in model_names:
    print(f'\n── {name} ──')
    print(classification_report(
        y_test, final_preds[name],
        target_names=CLASS_NAMES, zero_division=0
    ))

log('All done.')
print()
print('Output files:')
for f in sorted([f for f in os.listdir('.')
                 if f.startswith('outputs_') or f.endswith('.csv')]):
    print(f'  {f}')